In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import fetch_california_housing, load_diabetes, fetch_openml
from kernel_utils import *


# loading 


In [ ]:

# Datasets you want
datasets = ["cal_housing", "airfoil", "energy"]

rows = []
for name in datasets:
    # ---- load ----
    if name == "cal_housing":
        X, y = fetch_california_housing(return_X_y=True)
    elif name == "diabetes":
        X, y = load_diabetes(return_X_y=True)
    elif name == "airfoil":
        ds = fetch_openml(data_id=1507, as_frame=False, parser="auto")
        X, y = ds.data, ds.target
    elif name == "energy":
        ds = fetch_openml(data_id=1471, as_frame=False, parser="auto")
        X, y = ds.data, ds.target
    else:
        raise ValueError(name)

    # ---- y to float, 1D ----
    y = np.asarray(y).reshape(-1).astype(float)

    # ---- cap at 50k (take first 50k) ----
    if X.shape[0] > 50_000:
        X = X[:50_000]
        y = y[:50_000]

    # ---- remove NaNs ----
    mask = ~np.isnan(y) & ~np.isnan(X).any(axis=1)
    X, y = X[mask], y[mask]

    rows.append({
        "dataset": name,
        "n_samples": X.shape[0],
        "n_features": X.shape[1],
        "y_mean": float(y.mean()),
        "y_std": float(y.std()),
        "y_min": float(y.min()),
        "y_max": float(y.max()),
    })

df_info = pd.DataFrame(rows).sort_values("n_samples", ascending=False).reset_index(drop=True)
df_info

# Kernel functions

In [ ]:
#------------------==
# Kernel functions
#------------------==
def local_kernel(X1, X2, c):
    diff = X1[:, None, :] - X2[None, :, :]
    return np.prod(np.cos(0.5 * c * diff) ** 2, axis=-1)

def local_global_kernel(K, rho, q):
    return K + rho * (K ** q)


def mse(y_pred, y_true):
    return np.mean((y_pred - y_true)**2)


def compute_alpha(K, y, rho=0,eps = 1e-8):
    # compute the alpha solving  rho_ridge rgression with kernel matrix K and  labels y
    
    if rho !=0:
        n = K.shape[0]
        Krho = K + rho * np.eye(n)
    else:
        Krho = K
    
    try: # Krho invertible
        return np.linalg.solve(Krho, y)
    except np.linalg.LinAlgError: # Krho non invertible
        return np.linalg.pinv(Krho) @ y
    


def compute_alphafig1(K, y, rho=0,eps = 0):
    # compute the alpha solving  rho_ridge rgression with kernel matrix K and  labels y
    
    if rho !=0:
        n = K.shape[0]
        Krho = K + rho * np.eye(n)
    else:
        Krho = K + eps * np.eye(K.shape[0])
    
    try: # Krho invertible
        return np.linalg.solve(Krho, y)
    except np.linalg.LinAlgError: # Krho non invertible
        return np.linalg.pinv(Krho) @ y

# Grid 

In [ ]:
c_grid   = [0.1, 0.2, 0.4, 0.7, 1.0]
rho_grid = [0.01, 0.1,0.5, 1.0, 10]
powers = [3,4,5,6,7,8,9,10]
q_grid   = [int(p) for p in powers]

train_frac = 0.2
n_splits = 5
val_frac_of_remaining = 0.4

# Experiment

In [ ]:

# ------------------
# MAIN LOOP OVER DATASETS
# ------------------
all_rows = []
N_tresh = 25000
for dataset_name in datasets:

    print("Running:", dataset_name)

    # ---- load dataset ----
    if dataset_name == "cal_housing":
        X, y = fetch_california_housing(return_X_y=True)
    elif dataset_name == "diabetes":
        X, y = load_diabetes(return_X_y=True)
    elif dataset_name == "airfoil":
        ds = fetch_openml(data_id=1507, as_frame=False)
        X, y = ds.data, ds.target.astype(float)
    elif dataset_name == "energy":
        ds = fetch_openml(data_id=1471, as_frame=False)
        X, y = ds.data, ds.target.astype(float)


    if X.shape[0] > N_tresh:
        X = X[:N_tresh]
        y = y[:N_tresh]

    
    X = StandardScaler().fit_transform(X)

    n, d = X.shape
    rng = np.random.default_rng(0)

    # store per-split results
    results = []

    for split in range(n_splits):

        idx = rng.permutation(n)
        n_train = int(train_frac * n)

        train_idx = idx[:n_train]
        remaining_idx = idx[n_train:]

        n_val = int(val_frac_of_remaining * len(remaining_idx))
        val_idx = remaining_idx[:n_val]
        test_idx = remaining_idx[n_val:]

        X_train, y_train = X[train_idx], y[train_idx]
        X_val, y_val     = X[val_idx], y[val_idx]
        X_test, y_test   = X[test_idx], y[test_idx]

        # ------------------
        # Local/base ridgeless: tune c
        # ------------------
        best_local_c = None
        best_local_val = np.inf
        eps = 1e-10
        for c in c_grid:
            K_tr = local_kernel(X_train, X_train, c)
            alpha = compute_alphafig1(K_tr, y_train,eps)

            K_val = local_kernel(X_val, X_train, c)
            val_err = mse(K_val @ alpha, y_val)

            if val_err < best_local_val:
                best_local_val = val_err
                best_local_c = c

        # fit local with c_local*
        K_tr = local_kernel(X_train, X_train, best_local_c)
        alpha_local = compute_alphafig1(K_tr, y_train,eps)

        train_mse_local = mse(K_tr @ alpha_local, y_train)
        test_mse_local  = mse(local_kernel(X_test, X_train, best_local_c) @ alpha_local, y_test)

        # ------------------
        # LOCAL-GLOBAL ridgeless: tune (c,rho,q)
        # ------------------
        best = {"c": None, "rho": None, "q": None, "val_mse": np.inf}

        for c in c_grid:
            Kbase_tr  = local_kernel(X_train, X_train, c)
            Kbase_val = local_kernel(X_val, X_train, c)

            for rho in rho_grid:
                for q in q_grid:
                    K_tr = local_global_kernel(Kbase_tr, rho, q)
                    alpha = compute_alpha(K_tr, y_train)

                    K_val = local_global_kernel(Kbase_val, rho, q)
                    val_err = mse(K_val @ alpha, y_val)

                    if val_err < best["val_mse"]:
                        best = {"c": c, "rho": rho, "q": int(q), "val_mse": val_err}

        # fit LG with best params
        c_star, rho_star, q_star = best["c"], best["rho"], best["q"]

        Kbase_tr = local_kernel(X_train, X_train, c_star)
        K_tr = local_global_kernel(Kbase_tr, rho_star, q_star)
        alpha_lg = compute_alpha(K_tr, y_train)

        train_mse_lg = mse(K_tr @ alpha_lg, y_train)
        test_mse_lg  = mse(
            local_global_kernel(local_kernel(X_test, X_train, c_star), rho_star, q_star) @ alpha_lg,
            y_test
        )

        results.append({
            "split": split,

            "local_c": best_local_c,
            "local_val_mse": best_local_val,
            "local_train_mse": train_mse_local,
            "local_test_mse": test_mse_local,

            "lg_c": c_star,
            "lg_rho": rho_star,
            "lg_q": q_star,
            "lg_val_mse": best["val_mse"],
            "lg_train_mse": train_mse_lg,
            "lg_test_mse": test_mse_lg,
        })

    df = pd.DataFrame(results)

    # sizes (constant across splits given your splitting scheme)
    n_train = int(train_frac * n)
    n_rem = n - n_train
    n_val = int(val_frac_of_remaining * n_rem)
    n_test = n_rem - n_val

    # win rate: fraction of splits where LG beats local on test
    win_rate = float((df["lg_test_mse"] < df["local_test_mse"]).mean())

    # "best test" across splits + params that achieved it (descriptive)
    i_best_local = df["local_test_mse"].idxmin()
    i_best_lg    = df["lg_test_mse"].idxmin()

    all_rows.append({
        "dataset": dataset_name,
        "n": n,
        "d": d,
        "n_train": n_train,
        "n_val": n_val,
        "n_test": n_test,
        "train_frac": train_frac,
        "val_frac_remaining": val_frac_of_remaining,

        # --- average performance (main)
        "local_test_mean": df["local_test_mse"].mean(),
        "local_test_std": df["local_test_mse"].std(ddof=1),
        "lg_test_mean": df["lg_test_mse"].mean(),
        "lg_test_std": df["lg_test_mse"].std(ddof=1),

        # --- optional: train errors
        "local_train_mean": df["local_train_mse"].mean(),
        "local_train_std": df["local_train_mse"].std(ddof=1),
        "lg_train_mean": df["lg_train_mse"].mean(),
        "lg_train_std": df["lg_train_mse"].std(ddof=1),

        # --- win rate (LG better than local on test)
        "lg_win_rate": win_rate,

        # --- "typical" selected params (median over splits)
        "c_local_median": float(df["local_c"].median()),
        "c_lg_median": float(df["lg_c"].median()),
        "rho_lg_median": float(df["lg_rho"].median()),
        "q_lg_median": int(df["lg_q"].median()),

        # --- best test across splits (descriptive; we don't for selection)
        "local_test_best": float(df.loc[i_best_local, "local_test_mse"]),
        "local_test_best_c": float(df.loc[i_best_local, "local_c"]),

        "lg_test_best": float(df.loc[i_best_lg, "lg_test_mse"]),
        "lg_test_best_c": float(df.loc[i_best_lg, "lg_c"]),
        "lg_test_best_rho": float(df.loc[i_best_lg, "lg_rho"]),
        "lg_test_best_q": int(df.loc[i_best_lg, "lg_q"]),
    })

In [ ]:
final_table = pd.DataFrame(all_rows)
final_table

In [ ]:
final_table.columns

In [ ]:


# ------------------
# Save results
# ------------------
def save_results(df, filename, format="csv"):
    """
    Save DataFrame results to disk.
    
    format: "csv" or "pkl"
    """
    if format == "csv":
        df.to_csv(filename, index=False)
    elif format == "pkl":
        df.to_pickle(filename)
    else:
        raise ValueError("format must be 'csv' or 'pkl'")
    
    print(f"Results saved to {filename}")


# ------------------
# Load results
# ------------------
def load_results(filename, format="csv"):
    """
    Load saved results.
    """
    if format == "csv":
        df = pd.read_csv(filename)
    elif format == "pkl":
        df = pd.read_pickle(filename)
    else:
        raise ValueError("format must be 'csv' or 'pkl'")
    
    print(f"Results loaded from {filename}")
    return df

In [ ]:
save_results(df=final_table, filename = "regression_all.csv", format="csv")

# Table of results in latex

In [ ]:
import os
print(os.getcwd())

In [ ]:
import pandas as pd
import numpy as np

df_raw = pd.read_csv("regression_all.csv")

def latex_escape_text(s: str) -> str:
    return (str(s)
            .replace("\\", "\\textbackslash{}")
            .replace("_", "\\_")
            .replace("%", "\\%")
            .replace("&", "\\&")
            .replace("#", "\\#"))

def sci_to_latex(x, digits=2):
    x = float(x)
    if x == 0.0:
        return "0"
    ax = abs(x)
    if ax >= 1e4 or ax < 1e-2:
        exp = int(np.floor(np.log10(ax)))
        mant = x / (10 ** exp)
        return rf"{mant:.{digits}f}\times 10^{{{exp}}}"
    else:
        return rf"{x:.{digits}f}"

def pm_to_latex(mean, std, digits=2):
    return rf"${sci_to_latex(mean, digits)} \pm {sci_to_latex(std, digits)}$"

def to_latex_booktabs(df, column_format=None):
    tab = df.to_latex(index=False, escape=False, column_format=column_format)
    lines = tab.splitlines()
    hline_idx = [i for i, L in enumerate(lines) if "\\hline" in L]
    if len(hline_idx) >= 3:
        lines[hline_idx[0]] = "\\toprule"
        lines[hline_idx[1]] = "\\midrule"
        lines[hline_idx[-1]] = "\\bottomrule"
    lines = [L for L in lines if L.strip() != "\\hline"]
    return "\n".join(lines)

# -------------------------
# Table 1: dataset / splits (narrow)
# -------------------------
df_desc_small = pd.DataFrame({
    "Dataset": df_raw["dataset"].map(latex_escape_text),
    "$n$": df_raw["n"].astype(int),
    "$d$": df_raw["d"].astype(int),
    "$n_{\\mathrm{train}}$": df_raw["n_train"].astype(int),
    "$n_{\\mathrm{val}}$": df_raw["n_val"].astype(int),
    "$n_{\\mathrm{test}}$": df_raw["n_test"].astype(int),
}).sort_values("$n$", ascending=False).reset_index(drop=True)

latex_desc_small = to_latex_booktabs(df_desc_small, column_format="lrrrrr")

with open("metadata.tex", "w", encoding="utf-8") as f:
    f.write(latex_desc_small)

# -------------------------
# Table 2: performance (narrow)
# -------------------------
df_perf_small = pd.DataFrame({
    "Dataset": df_raw["dataset"].map(latex_escape_text),
    r"$k$": [pm_to_latex(m, s, 2) for m, s in zip(df_raw["local_test_mean"], df_raw["local_test_std"])],
    r"k_{LG}":    [pm_to_latex(m, s, 2) for m, s in zip(df_raw["lg_test_mean"], df_raw["lg_test_std"])],
}).sort_values("Dataset").reset_index(drop=True)

latex_perf_small = to_latex_booktabs(df_perf_small, column_format="llll")

with open("regressionMSE.tex", "w", encoding="utf-8") as f:
    f.write(latex_perf_small)

print("Saved: metadata.tex, regressionMSE.tex")

In [ ]:
print(latex_perf_small)

In [ ]:
print(latex_desc_small)

In [ ]:
import pandas as pd
import numpy as np

df_raw = pd.read_csv("regression_all.csv")

def latex_escape_text(s: str) -> str:
    return (str(s)
            .replace("\\", "\\textbackslash{}")
            .replace("_", "\\_")
            .replace("%", "\\%")
            .replace("&", "\\&")
            .replace("#", "\\#"))

def sci_to_latex(x, digits=2):
    x = float(x)
    if x == 0.0:
        return "0"
    ax = abs(x)
    if ax >= 1e4 or ax < 1e-2:
        exp = int(np.floor(np.log10(ax)))
        mant = x / (10 ** exp)
        return rf"{mant:.{digits}f}\times 10^{{{exp}}}"
    else:
        return rf"{x:.{digits}f}"

def float_to_math(x, digits=2):
    return rf"${sci_to_latex(x, digits)}$"

def to_latex_booktabs(df, column_format=None):
    tab = df.to_latex(index=False, escape=False, column_format=column_format)
    lines = tab.splitlines()
    hline_idx = [i for i, L in enumerate(lines) if "\\hline" in L]
    if len(hline_idx) >= 3:
        lines[hline_idx[0]] = "\\toprule"
        lines[hline_idx[1]] = "\\midrule"
        lines[hline_idx[-1]] = "\\bottomrule"
    lines = [L for L in lines if L.strip() != "\\hline"]
    return "\n".join(lines)



# ---------------------------------------
# Option B: include Local best too (wider)
# ---------------------------------------
df_best_both = pd.DataFrame({
    "Dataset": df_raw["dataset"].map(latex_escape_text),
    r"$k$: Best MSE": df_raw["local_test_best"].map(lambda x: float_to_math(x, 2)),
    "$c^*_{\\mathrm{base}}$": df_raw["local_test_best_c"].map(lambda x: float_to_math(x, 2)),
    r"$k_{LG}$: Best MSE": df_raw["lg_test_best"].map(lambda x: float_to_math(x, 2)),
    "$c^*$": df_raw["lg_test_best_c"].map(lambda x: float_to_math(x, 2)),
    "$\\rho^*$": df_raw["lg_test_best_rho"].map(lambda x: float_to_math(x, 2)),
    "$q^*$": df_raw["lg_test_best_q"].astype(int),
}).sort_values("Dataset").reset_index(drop=True)

latex_best_both = to_latex_booktabs(df_best_both, column_format="lrrrrrr")

with open("params_best_MSE_Test.tex", "w", encoding="utf-8") as f:
    f.write(latex_best_both)

print("Saved: table_best_params_regression.tex")

In [ ]:
print(latex_best_both)

In [ ]:
import pandas as pd
import numpy as np

df_raw = pd.read_csv("regression_all.csv")

def latex_escape(s):
    return (str(s)
            .replace("\\", "\\textbackslash{}")
            .replace("_", "\\_")
            .replace("%", "\\%")
            .replace("&", "\\&")
            .replace("#", "\\#"))

def sci_sig(x, sig):
    """
    Scientific notation with 'sig' significant digits.
    Always returns mantissa × 10^{exp}
    """
    x = float(x)

    if x == 0:
        return "0"

    sign = "-" if x < 0 else ""
    x = abs(x)

    exp = int(np.floor(np.log10(x)))
    mant = x / (10**exp)

    # round to significant digits
    mant = round(mant, sig - 1)

    # if rounding makes mant = 10.0
    if mant >= 10:
        mant /= 10
        exp += 1

    mant_str = f"{mant:.{sig-1}f}"

    return rf"{sign}{mant_str}\cdot 10^{{{exp}}}"

def pm(mean, std):
    mean_str = sci_sig(mean, sig=2)  # 2 significant digits
    std_str  = sci_sig(std, sig=1)   # 1 significant digit
    return rf"${mean_str} \pm {std_str}$"

def to_booktabs(df, column_format):
    tab = df.to_latex(index=False, escape=False, column_format=column_format)
    lines = tab.splitlines()
    h = [i for i,L in enumerate(lines) if L.strip()=="\\hline"]
    if len(h) >= 3:
        lines[h[0]] = "\\toprule"
        lines[h[1]] = "\\midrule"
        lines[h[-1]] = "\\bottomrule"
    lines = [L for L in lines if L.strip()!="\\hline"]
    return "\n".join(lines)

# ---- build performance table ----
df_perf = pd.DataFrame({
    "Dataset": df_raw["dataset"].map(latex_escape),
    r"$k_L$ MSE train": [pm(m,s) for m,s in zip(df_raw["local_train_mean"], df_raw["local_train_std"])],
    r"$k_{LG}$ MSE train":    [pm(m,s) for m,s in zip(df_raw["lg_train_mean"], df_raw["lg_train_std"])],
}).sort_values("Dataset").reset_index(drop=True)

latex_perf = to_booktabs(df_perf, column_format="lllll")

with open("regressionMSEtrain.tex", "w", encoding="utf-8") as f:
    f.write(latex_perf)

print(latex_perf)
print("Saved: regressionMSEtrain.tex")